# Пайплайн обучения моделей для типизации режимов скважин

**Этапы:**
1. Загрузка и объединение данных из .txt файлов
2. Очистка данных (удаление выключенных скважин, выбросов)
3. Формирование скользящих окон (ЭЦН: 60 точек, ШГН: 120 точек)
4. Обучение LSTM автоэнкодеров
5. Построение латентного пространства
6. Кластеризация
7. Получение типовых профилей

**Данные:**
- ЭЦН: скважина 133 (периодическая работа, стабильный поток)
- ШГН: 2 скважины (циклическая работа ~7 мин: нефть → вода → газ)

In [15]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [16]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)

## Этап 1: Загрузка данных из .txt файлов

Структура данных:
- Дата, Время, 9 параметров влагомера, Номер отвода (игнорируем)
- Разделитель: табуляция
- Интервал: 10 секунд

In [17]:
COLUMN_NAMES = [
    'date', 'time',
    'us_center', 'us_periph',
    'gas_center', 'gas_periph',
    'temp',
    'water_center', 'water_periph',
    'outlet_num',
    'gas_integral', 'water_integral'
]

FEATURE_COLUMNS = [
    'us_center', 'us_periph',
    'gas_center', 'gas_periph',
    'temp',
    'water_center', 'water_periph',
    'gas_integral', 'water_integral'
]

DATA_ROOT = Path('../data')

WELLS = {
    'ecn': ['скважина 133 ЭЦН'],
    'shgn': ['скважина 134 ШГН', 'скважина 135 ШГН']
}

In [18]:
def load_well_data(well_folder):
    txt_files = sorted(well_folder.glob("*.txt"))

    if not txt_files:
        print(f"Нет .txt файлов в {well_folder}")
        return pd.DataFrame()

    print(f"Найдено {len(txt_files)} файлов в {well_folder.name}")

    dfs = []
    for file in txt_files:
        try:
            df = pd.read_csv(file, sep="\t", names=COLUMN_NAMES, encoding="utf-8")
            df["timestamp"] = pd.to_datetime(df["date"] + " " + df["time"], format="%Y.%m.%d %H:%M:%S")
            df = df.drop(columns=["date", "time"])
            df["well"] = well_folder.name
            dfs.append(df)
        except Exception as e:
            print(f"Ошибка чтения {file.name}: {e}")

    if not dfs:
        return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.sort_values("timestamp").reset_index(drop=True)

    print(f"  Загружено {len(combined):,} записей")
    print(f"  Период: {combined['timestamp'].min()} - {combined['timestamp'].max()}")

    return combined


In [21]:
ecn_dfs = []
for well_name in WELLS['ecn']:
    well_path = DATA_ROOT / well_name
    if well_path.exists():
        df = load_well_data(well_path)
        if not df.empty:
            ecn_dfs.append(df)

df_ecn = pd.concat(ecn_dfs, ignore_index=True) if ecn_dfs else pd.DataFrame()

print(f"\n{'='*60}")
print(f"ЭЦН: {len(df_ecn):,} записей")
print(f"{'='*60}")

Найдено 16 файлов в скважина 133 ЭЦН
  Загружено 51,908 записей
  Период: 2021-11-01 08:59:50 - 2021-12-01 07:54:30

ЭЦН: 51,908 записей


In [22]:
shgn_dfs = []
for well_name in WELLS['shgn']:
    well_path = DATA_ROOT / well_name
    if well_path.exists():
        df = load_well_data(well_path)
        if not df.empty:
            shgn_dfs.append(df)

df_shgn = pd.concat(shgn_dfs, ignore_index=True) if shgn_dfs else pd.DataFrame()
before_filter = len(df_shgn)
df_shgn = df_shgn[df_shgn['timestamp'] >= '2021-01-01']
removed = before_filter - len(df_shgn)

print("\nФильтрация некорректных дат (до 2021 года)")
print(f"  Удалено: {removed} записей")
print(f"  Осталось: {len(df_shgn):,} записей")
print(f"  Новый период: {df_shgn['timestamp'].min()} → {df_shgn['timestamp'].max()}")
print(f"\n{'='*60}")
print(f"ШГН: {len(df_shgn):,} записей")
print(f"{'='*60}")

Найдено 9 файлов в скважина 134 ШГН
  Загружено 83,487 записей
  Период: 1970-03-10 06:25:40 - 2021-11-30 23:54:20
Найдено 6 файлов в скважина 135 ШГН
  Загружено 47,631 записей
  Период: 2021-10-19 23:59:30 - 2021-11-30 23:54:20

Фильтрация некорректных дат (до 2021 года)
  Удалено: 1 записей
  Осталось: 131,117 записей
  Новый период: 2021-10-19 23:59:30 → 2021-11-30 23:54:20

ШГН: 131,117 записей


## Сохранение очищенных данных

In [23]:
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ecn_path = PROCESSED_DIR / 'ecn_clean.csv'
shgn_path = PROCESSED_DIR / 'shgn_clean.csv'

df_ecn.to_csv(ecn_path, index=False)
df_shgn.to_csv(shgn_path, index=False)

print("\n" + "="*70)
print("СОХРАНЕНИЕ ДАННЫХ")
print("="*70)
print("\nСохранено:")
print(f"  {ecn_path.name}: {len(df_ecn):,} записей")
print(f"  {shgn_path.name}: {len(df_shgn):,} записей")


СОХРАНЕНИЕ ДАННЫХ

Сохранено:
  ecn_clean.csv: 51,908 записей
  shgn_clean.csv: 131,117 записей


1. Базовая статистика
Описательные статистики по каждому параметру (mean, std, min, max)
Проверка пропусков и дубликатов
Распределение значений (гистограммы)
2. Временные графики
Все 9 параметров по времени для каждого типа насоса
Выявление паттернов:
ЭЦН: периодичность включения/выключения
ШГН: циклы ~7 мин (нефть → вода → газ)
3. Анализ состояний скважин
Детектирование выключенных периодов (константные значения)
Статистика: сколько времени работала/не работала
Удаление выключенных периодов
4. Корреляционный анализ
Heatmap корреляций между параметрами
Выявление взаимосвязей (например, газ vs обводнённость)
5. Анализ циклов ШГН
Автокорреляция для подтверждения периода ~7 мин
Визуализация типичного цикла
6. Выбросы и аномалии
Box plots для каждого параметра
Выявление и обработка выбросов
7. Подготовка к нормализации
Диапазоны значений каждого параметра
Выбор метода нормализации (MinMax vs Standard)